<a href="https://colab.research.google.com/github/IsaacVic-Dark/Gemma-FineTune-colab-notebooks/blob/main/Train_InsuOps_Model_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
from google.colab import files

# Upload quotation_training_data.json
uploaded = files.upload()

with open('quotation_training_data.json', 'r') as f:
    quotation_dataset = json.load(f)

# Optional: rewrite nicely formatted JSON
with open('quotation_training_data.json', 'w') as json_file:
    json.dump(quotation_dataset, json_file, indent=4)

print(f"Data loaded! Total Q&A pairs: {len(quotation_dataset)}")

Saving quotation_training_data.json to quotation_training_data.json
Data loaded! Total Q&A pairs: 31


In [ ]:
from datasets import Dataset
import json

def format_prompt(item):
    return {
        "text": (
            "<start_of_turn>user\n "
            f"{item['question']}"
            "<end_of_turn><start_of_turn>model\n "
            f"{item['answer']}"
        )
    }

file = json.load(open("quotation_training_data.json", "r"))
formatted_data = [format_prompt(item) for item in file]

dataset = Dataset.from_list(formatted_data)
dataset.save_to_disk("quotation_training_dataset")

print(dataset[0])

Saving the dataset (0/1 shards):   0%|          | 0/31 [00:00<?, ? examples/s]

{'text': "<start_of_turn>user\n How does the quotation process work in this system?<end_of_turn><start_of_turn>model\n The quotation process in our system follows these steps: First, a broker creates a Quote Request (prefixed 'QR') by submitting client details, product category, coverage requirements, and one or more insurable items. The system then finds all active insurance products matching the requested category. For each matching product, the system decides whether to auto-calculate a quote or send it for manual underwriting. Once quotes are generated, they are stored as Underwriter Quotes (prefixed 'UQ') linked back to the original Quote Request. The Quote Request status is then updated to 'quoted'."}


In [ ]:
%%capture
import os

# Clean up conflicting packages first
!pip uninstall -y torchao
!pip uninstall -y unsloth unsloth_zoo peft bitsandbytes accelerate xformers trl cut_cross_entropy

# Colab-specific Unsloth installation
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# !pip install torchao>=0.16.0

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# from unsloth import FastModel

# model, tokenizer = FastModel.from_pretrained(
#     model_name="/content/drive/MyDrive/gemma-3",
#     load_in_4bit=True,
#     max_seq_length=2048
# )

# After running this, switch to inference mode:
# FastModel.for_inference(model)

In [ ]:
from unsloth import FastModel
import torch

model_name = "unsloth/gemma-3-1b-it"

model, tokenizer = FastModel.from_pretrained(
    model_name=model_name,
    dtype=None,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    max_seq_length=2048,
)

print(type(model), type(tokenizer))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

<class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM'> <class 'transformers.models.gemma.tokenization_gemma.GemmaTokenizer'>


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
    use_rslora=False,
    loftq_config=None,
    bias="none",
    random_state=3407,
)

print("PEFT model ready")

PEFT model ready


In [ ]:
# set up the training engine

from trl import SFTTrainer, SFTConfig
from datasets import Dataset

dataset = Dataset.load_from_disk("quotation_training_dataset")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    dataset_text_field="text",
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        max_steps=40,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        max_length=None,
        padding_free=False,  # explicitly disable Unsloth's default
    )
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n"
)

Map (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

In [ ]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

"<bos><start_of_turn>user\n How does the quotation process work in this system?<end_of_turn><start_of_turn>model\n The quotation process in our system follows these steps: First, a broker creates a Quote Request (prefixed 'QR') by submitting client details, product category, coverage requirements, and one or more insurable items. The system then finds all active insurance products matching the requested category. For each matching product, the system decides whether to auto-calculate a quote or send it for manual underwriting. Once quotes are generated, they are stored as Underwriter Quotes (prefixed 'UQ') linked back to the original Quote Request. The Quote Request status is then updated to 'quoted'."

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31 | Num Epochs = 10 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,045,760 of 1,012,931,712 (1.29% trained)


Step,Training Loss
1,5.632970
2,5.307568
3,5.303476
4,4.726360
5,4.718298
6,3.710076
7,3.362207
8,3.632846
9,3.301917
10,3.040049


TrainOutput(global_step=40, training_loss=2.5321117103099824, metrics={'train_runtime': 623.5717, 'train_samples_per_second': 0.513, 'train_steps_per_second': 0.064, 'total_flos': 161698030334208.0, 'train_loss': 2.5321117103099824})

In [ ]:
FastModel.for_inference(model)

messages= [{
 "role":"user",
 "content":[{
     "type":"text",
     "text": "How does the quotation process work in this system?"
 }]
}]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenizer=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=text,
    max_new_tokens=256,
    use_cache=True,
    do_sample=True,
    temperature=1.0,
    top_k=64,
    top_p=0.95
)

response = tokenizer.batch_decode(outputs)[0]
print(response)

In [ ]:
# This saves the trainend weights'not full model since we are using LoRA'

from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/Insuops_lora_model_v2"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Saved successfully!")